<a href="https://colab.research.google.com/github/KalindiGosine/qpix-digital/blob/main/dev_journals/kgosine_2026_02_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Chip Network Simulation

### Requirements for presentation:
* estimate of required computer resources
* rate of clk cycles simualted / sec/ CPU
* diagram of chip definition
* diagram of network design

### ZeroMQ
Push-pull (pipeline) which connects nodes in a fan-out.fan-in pattern. It is a parallel task distribution and collection pattern which defines a "parallelized pipeline".

### Digital Twin
While a simualtion can only account for a set of pre-determined cases and requires some kind of input to actualize it to an obkect and a state, a digital twin proves its input automatically using a collection of sensors. A digital twin includes simualtion processes but the system is larger overall.


### Example RTL Design

5MHz digital clock speed. A chip generated 64-bit data packets from a pseudo-random number generator. The packets are sent in bursts with the time between them also selected from a random number generator. The intervals between bursts is bounded by a max_gap_cycles and min_gap_cycles. A chip can also send a random number of packets back-to-back bounded by min_burst_pkts and max_burst_pkts.

In [ ]:
// Verilog RTL for single chip

module packet_gen_burst #(
    parameter integer SEED = 32'h1234_5678,

    // Random gap between bursts (cycles)
    parameter integer MIN_GAP_CYCLES  = 5_000,     // 1 ms @ 5 MHz
    parameter integer MAX_GAP_CYCLES  = 50_000,    // 10 ms @ 5 MHz

    // Random burst length (packets per burst)
    parameter integer MIN_BURST_PKTS  = 1,
    parameter integer MAX_BURST_PKTS  = 8
)(
    input  wire        clk,
    input  wire        rst_n,

    // Output packet interface (valid/ready)
    output reg  [63:0] pkt_data,
    output reg         pkt_valid,
    input  wire        pkt_ready
);

    // ---------------------------
    // 32-bit Linear Feedback Shift Register (LFSR) for randomness
    // i.e. psuedo random number generator
    // ---------------------------
    reg [31:0] lfsr;

    function [31:0] lfsr_next(input [31:0] x);
        begin
            // x^32 + x^22 + x^2 + x^1 + 1
            lfsr_next = {x[30:0], x[31] ^ x[21] ^ x[1] ^ x[0]};
        end
    endfunction

    // ---------------------------
    // Helpers to pick random ranges for the time gap between bursts
    // & the number of packets in a burst
    //---------------------------
    function [31:0] pick_gap(input [31:0] r);
        integer span;
        begin
            span = (MAX_GAP_CYCLES - MIN_GAP_CYCLES + 1);
            pick_gap = MIN_GAP_CYCLES + (r % span);
        end
    endfunction

    function [31:0] pick_burst_len(input [31:0] r);
        integer span;
        begin
            span = (MAX_BURST_PKTS - MIN_BURST_PKTS + 1);
            pick_burst_len = MIN_BURST_PKTS + (r % span);
        end
    endfunction

    // ---------------------------
    // State and counters
    // Define three possible states for the chip to be in
    // ---------------------------

    // ---------------------------
    // state = 0 "waiting before sending out the next burst"
    // gap_countdown (random number) decrements every clock
    // ----------------------------
    localparam WAIT_GAP  = 2'd0;

    // ---------------------------
    // state = 1 "start sending a burst and decide how long it is"
    // a random burst length is chosen
    // immediately transistion to state=2, send_pkt
    // ----------------------------
    localparam START_BURST = 2'd1;

    // ---------------------------
    // state = 2 "actively sending out data packet(s)"
    // pkt_valid must be asserted
    // when pkt_valid && pkt_ready are high, a 64-bit packet is accepted as valid
    // burst_remaining decremements
    // when burst_remaining counts down to zero, a new random gap is chosen
    // transistion to state=0 (wait)
    // ----------------------------
    localparam SEND_PKT  = 2'd2;

    // these two bits specify which of the three states the chip is in
    reg [1:0]  state;
    // holds the number of cycles remaining in the wait period
    reg [31:0] gap_countdown;
    // holds the number of packets remaining to go out in the burst
    reg [31:0] burst_remaining;

    // For building 64-bit packet
    // declare two 32-bit registers because the random number generator only creates 32 random bits
    // ie advance the lfsr once, get 32 random bits, store in r1
    // then advance lfsr again, get another 32 random bits, store in r2
    // concatenate to form a 64-bit packet
    reg [31:0] r1, r2;

    // Fir Rule: data transfer only happens in a clk cycle if and only if valid and ready are both 1 on the rising edge
    // pkt_ready refers to downstream ready (comes from higher level software)
    // pkt_valid refers to packet ready (comes from this RTL)
    // used for interchip data transfer agreement
    wire fire = pkt_valid && pkt_ready;

    always @(posedge clk) begin
        if (!rst_n) begin
            lfsr            <= SEED;
            // initialize a 64-bit data packet of all 0's
            pkt_data        <= 64'd0;
            // intialize pkt_valid as 0 (the packet is not ready to be sent)
            pkt_valid       <= 1'b0;
            // intialize the number of gaps between sends as the minimum
            gap_countdown   <= MIN_GAP_CYCLES;
            // initialize the number of packets in a burst remaining to 0's
            burst_remaining <= 32'd0;
            // intialize the registers to hold the random numbers generated to 0's
            r1              <= 32'd0;
            r2              <= 32'd0;
            // intialize chip state as state=0, waiting to send a packet
            state           <= WAIT_GAP;
        end else begin
            case (state)

                // ---------------------------
                // Wait until it's time to start a burst
                // ---------------------------
                WAIT_GAP: begin
                    pkt_valid <= 1'b0;

                    if (gap_countdown != 0) begin
                        gap_countdown <= gap_countdown - 1;
                        lfsr <= lfsr_next(lfsr);
                    end else begin
                        state <= START_BURST;
                    end
                end

                // ---------------------------
                // Choose random burst length
                // ---------------------------
                START_BURST: begin
                    // advance randomness
                    lfsr <= lfsr_next(lfsr);

                    // decide how many packets in this burst
                    burst_remaining <= pick_burst_len(lfsr);

                    state <= SEND_PKT;
                end

                // ---------------------------
                // Send packets back-to-back (handshake controlled)
                // ---------------------------
                SEND_PKT: begin
                    // If not currently asserting valid, generate a new packet
                    if (!pkt_valid) begin
                        // Use two LFSR steps to make 64 bits
                        r1   <= lfsr_next(lfsr);
                        r2   <= lfsr_next(lfsr_next(lfsr));
                        lfsr <= lfsr_next(lfsr_next(lfsr));

                        pkt_data  <= {r1, r2};
                        pkt_valid <= 1'b1;
                    end else if (fire) begin
                        // Packet accepted
                        pkt_valid <= 1'b0;

                        // decrement burst count
                        if (burst_remaining > 0)
                            burst_remaining <= burst_remaining - 1;

                        // If that was the last packet of the burst, schedule next gap
                        if (burst_remaining <= 1) begin
                            lfsr          <= lfsr_next(lfsr);
                            gap_countdown <= pick_gap(lfsr);
                            state         <= WAIT_GAP;
                        end
                        // else stay in SEND_PKT and emit next packet immediately
                    end
                end

                default: begin
                    state <= WAIT_GAP;
                end
            endcase
        end
    end

endmodule

### Combination wire for packet acceptance
| pkt_valid       | pkt_ready | fire | meaning|
| ----------- | ----------- | ---------- | --------|
| 0 | 0 | 0 | No packet, no transfer |
| 1  | 0 | 0 | Packet prepared, downstream stalled |
|0 |1 | 0 | Downstream ready, packet not prepared |
| 1 | 1| 1| packet prepared, downstream ready, packet transferred|

If pkt_ready==0, downstream stalled, pkt_valid reamins high and packet is retried next cycle (no data loss)

this wire will tell when to copy pkt_data into C memory and send over ZeroMQ. pkt_ready (downstream signal) will be known from the outbound software queue space. pkt_valid comes from chip RTL.

![](https://drive.google.com/uc?export=view&id=1XzviploI9LR48JOLeA6C-HyOvYkQL19c)

